In [1]:
# Install the dataretrieval package for accessing USGS water data
!pip install dataretrieval

# I want to use SQL queries to explore the data, so I'll install duckdb-python, which allows me to run SQL queries on pandas DataFrames in-memory.
!pip install duckdb

In [2]:
# refactor above into code for a src/utils.py file
# This will be used across multiple notebooks and scripts, so it's best to have a single source of truth
# for these mappings.
# We can add some pytests to ensure I did the mappings correctly and to prevent future regressions if this file
# is edited.
# Typos happen!
from typing import Optional

# Defined by https://nvlpubs.nist.gov/nistpubs/Legacy/FIPS/fipspub5-2.pdf
# Single source of truth for State Names, 2-digit FIPS codes, and USGS WQP Query Codes
US_STATE_FIPS_TABLE = [
    {"state_name": "Alabama", "fips_code": "01", "wqp_code": "US:01"},
    {"state_name": "Alaska", "fips_code": "02", "wqp_code": "US:02"},
    {"state_name": "Arizona", "fips_code": "04", "wqp_code": "US:04"},
    {"state_name": "Arkansas", "fips_code": "05", "wqp_code": "US:05"},
    {"state_name": "California", "fips_code": "06", "wqp_code": "US:06"},
    {"state_name": "Colorado", "fips_code": "08", "wqp_code": "US:08"},
    {"state_name": "Connecticut", "fips_code": "09", "wqp_code": "US:09"},
    {"state_name": "Delaware", "fips_code": "10", "wqp_code": "US:10"},
    {"state_name": "District of Columbia", "fips_code": "11", "wqp_code": "US:11"},
    {"state_name": "Florida", "fips_code": "12", "wqp_code": "US:12"},
    {"state_name": "Georgia", "fips_code": "13", "wqp_code": "US:13"},
    {"state_name": "Hawaii", "fips_code": "15", "wqp_code": "US:15"},
    {"state_name": "Idaho", "fips_code": "16", "wqp_code": "US:16"},
    {"state_name": "Illinois", "fips_code": "17", "wqp_code": "US:17"},
    {"state_name": "Indiana", "fips_code": "18", "wqp_code": "US:18"},
    {"state_name": "Iowa", "fips_code": "19", "wqp_code": "US:19"},
    {"state_name": "Kansas", "fips_code": "20", "wqp_code": "US:20"},
    {"state_name": "Kentucky", "fips_code": "21", "wqp_code": "US:21"},
    {"state_name": "Louisiana", "fips_code": "22", "wqp_code": "US:22"},
    {"state_name": "Maine", "fips_code": "23", "wqp_code": "US:23"},
    {"state_name": "Maryland", "fips_code": "24", "wqp_code": "US:24"},
    {"state_name": "Massachusetts", "fips_code": "25", "wqp_code": "US:25"},
    {"state_name": "Michigan", "fips_code": "26", "wqp_code": "US:26"},
    {"state_name": "Minnesota", "fips_code": "27", "wqp_code": "US:27"},
    {"state_name": "Mississippi", "fips_code": "28", "wqp_code": "US:28"},
    {"state_name": "Missouri", "fips_code": "29", "wqp_code": "US:29"},
    {"state_name": "Montana", "fips_code": "30", "wqp_code": "US:30"},
    {"state_name": "Nebraska", "fips_code": "31", "wqp_code": "US:31"},
    {"state_name": "Nevada", "fips_code": "32", "wqp_code": "US:32"},
    {"state_name": "New Hampshire", "fips_code": "33", "wqp_code": "US:33"},
    {"state_name": "New Jersey", "fips_code": "34", "wqp_code": "US:34"},
    {"state_name": "New Mexico", "fips_code": "35", "wqp_code": "US:35"},
    {"state_name": "New York", "fips_code": "36", "wqp_code": "US:36"},
    {"state_name": "North Carolina", "fips_code": "37", "wqp_code": "US:37"},
    {"state_name": "North Dakota", "fips_code": "38", "wqp_code": "US:38"},
    {"state_name": "Ohio", "fips_code": "39", "wqp_code": "US:39"},
    {"state_name": "Oklahoma", "fips_code": "40", "wqp_code": "US:40"},
    {"state_name": "Oregon", "fips_code": "41", "wqp_code": "US:41"},
    {"state_name": "Pennsylvania", "fips_code": "42", "wqp_code": "US:42"},
    {"state_name": "Rhode Island", "fips_code": "44", "wqp_code": "US:44"},
    {"state_name": "South Carolina", "fips_code": "45", "wqp_code": "US:45"},
    {"state_name": "South Dakota", "fips_code": "46", "wqp_code": "US:46"},
    {"state_name": "Tennessee", "fips_code": "47", "wqp_code": "US:47"},
    {"state_name": "Texas", "fips_code": "48", "wqp_code": "US:48"},
    {"state_name": "Utah", "fips_code": "49", "wqp_code": "US:49"},
    {"state_name": "Vermont", "fips_code": "50", "wqp_code": "US:50"},
    {"state_name": "Virginia", "fips_code": "51", "wqp_code": "US:51"},
    {"state_name": "Washington", "fips_code": "53", "wqp_code": "US:53"},
    {"state_name": "West Virginia", "fips_code": "54", "wqp_code": "US:54"},
    {"state_name": "Wisconsin", "fips_code": "55", "wqp_code": "US:55"},
    {"state_name": "Wyoming", "fips_code": "56", "wqp_code": "US:56"},
    # Key Territories
    {"state_name": "American Samoa", "fips_code": "60", "wqp_code": "US:60"},
    {"state_name": "Guam", "fips_code": "66", "wqp_code": "US:66"},
    {"state_name": "Northern Mariana Islands", "fips_code": "69", "wqp_code": "US:69"},
    {"state_name": "Puerto Rico", "fips_code": "72", "wqp_code": "US:72"},
    {"state_name": "Virgin Islands", "fips_code": "78", "wqp_code": "US:78"}
]

# --- Helper Functions ---
# We normalize the input to be case-insensitive and to handle potential whitespace issues, 
# but we could add more robust error handling if needed (e.g., for typos or invalid inputs).
# TODO: Add error handling for invalid inputs (e.g., typos, non-existent states, etc.) if needed.
# TODO: Add pytests for these functions to ensure correct mappings and to prevent future regressions
# if this file is edited, and to ensure error handling works as expected.

def get_wqp_code(state_name: str) -> Optional[str]:
    """Retrieve the WQP query code (e.g., 'US:53') using the full state name."""
    match = next((row for row in US_STATE_FIPS_TABLE if row["state_name"].lower() == state_name.lower()), None)
    return match["wqp_code"] if match else None

def get_state_from_wqp(wqp_code: str) -> Optional[str]:
    """Retrieve the full state name using the WQP query code (e.g., 'US:53')."""
    match = next((row for row in US_STATE_FIPS_TABLE if row["wqp_code"].upper() == wqp_code.upper()), None)
    return match["state_name"] if match else None

def get_fips_from_state(state_name: str) -> Optional[str]:
    """Retrieve the 2-digit FIPS code using the full state name."""
    match = next((row for row in US_STATE_FIPS_TABLE if row["state_name"].lower() == state_name.lower()), None)
    return match["fips_code"] if match else None

def get_state_from_fips(fips_code: str) -> Optional[str]:
    """Retrieve the full state name using the 2-digit FIPS code."""
    # Ensure it's a string and padded to 2 digits just in case an int was passed
    fips_str = str(fips_code).zfill(2)
    match = next((row for row in US_STATE_FIPS_TABLE if row["fips_code"] == fips_str), None)
    return match["state_name"] if match else None

def get_wqp_statecode_from_state(state_name: str) -> Optional[str]:
    """Retrieve the WQP query code (e.g., 'US:53') using the full state name."""
    return get_wqp_code(state_name)

# sanity asserts for development - these will be replaced by pytests in a separate test file
# but for now we can just run them here to ensure the mappings are correct.
assert get_state_from_wqp("US:53") == "Washington", f"Expected 'Washington', got '{get_state_from_wqp('US:53')}'"
assert get_fips_from_state("Washington") == "53", f"Expected '53', got '{get_fips_from_state('Washington')}'"
assert get_state_from_fips("53") == "Washington", f"Expected 'Washington', got '{get_state_from_fips('53')}'"
assert get_wqp_statecode_from_state("Washington") == "US:53", f"Expected 'US:53', got '{get_wqp_statecode_from_state('Washington')}'"


# Housekeeping 

Ensure we have directories to put things in!


In [3]:
import os
# Create local directory structure if they don't exist
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/bronze", exist_ok=True)
os.makedirs("data/quarantine", exist_ok=True)

# First dataset pull

I'm exploring the WQP project data. Their API [(documentation here)](https://www.waterqualitydata.us/webservices_documentation/) is pretty flexible and allows you to specify a wide range of parameters to filter the results.

To query the data, you can use POST or you can use the (dataretrieval package)[https://github.com/DOI-USGS/dataretrieval-python]. It provides a convenient interface to access this data directly from Python, and it will return the results as a pandas DataFrame, which is great for analysis and manipulation. I was figuring I'd need to build my own logic for the rest query -> dataframe pipeline, so this is handy. We just need to import [`wqp`](https://github.com/DOI-USGS/dataretrieval-python/blob/main/dataretrieval/wqp.py) from `dataretrieval` in order to query it.

I did run into some early issues with the query parameters. The dataretrieval.wqp.what_sites() queries require the statecode to be formatted as a 5-digit code, e.g. 'US:53' for Washington. I built a bit of tooling to help out, which takes the state name 'Washington' and returns the formatted code.

## First query: get all sites, then get the results for the sites

Let's pull ALL water temperature and pH records for Washington state (US:53) from 2015 to 2025. This will include all stream monitoring sites in the state, and all records of those two parameters over for a 10-year span. This will yield a massive, messy dataset... hopefully!

The first step is to get the sampling sites themselves, using get_sites(). We'll receive two items in return: a pandas Dataframe and a "dataretrieval.wqp.WQP_Metadata" object.

In [4]:
# Params for the WQP query for Washington stream monitoring sites
state_name = "Washington"
fips_code = get_wqp_statecode_from_state(state_name)
site_type = 'Stream' # We want stream monitoring sites, not lakes or groundwater wells

# Once we have the sites, we'll be able to query them for the data of interest: temperature and pH data from 2015 to 2025
# To read the docs on the get_results() function and its params, see the [wqp.py](https://github.com/DOI-USGS/dataretrieval-python/blob/main/dataretrieval/wqp.py) on the github repo.

# One or more case-sensitive characteristic names, separated by semicolons (https://www.waterqualitydata.us/public_srsnames/).
characteristics = ['Temperature, water', 'pH'] # We will need to format these for the get_results() query

# startDateLo : string
# Date of earliest desired data-collection activity, expressed as 'MM-DD-YYYY'
# startDateHi : string
# Date of last desired data-collection activity, expressed as 'MM-DD-YYYY'
start_date = '01-01-2025' # earlier I had the date format as 'YYYY-MM-DD', but the docs say it should be 'MM-DD-YYYY'. Oops!
end_date = '01-01-2025'

# The returns for the get_results() function are a pandas DataFrame and a WQP_Metadata object, which contains metadata about the query.
# This matches the get_sites() function.
#   Returns
#     -------
#     df : ``pandas.DataFrame``
#         Formatted data returned from the API query.
#     md : :obj:`dataretrieval.utils.Metadata`
#         Custom ``dataretrieval`` metadata object pertaining to the query.

# Retrieving the site data and save to Raw

In [5]:
from dataretrieval import wqp

print(f"Fetching stream monitoring sites in {state_name}...")

# Step 1: Find the stream sites in the given state
# Okay, so the sites variables is a pandas Dataframe and a "dataretrieval.wqp.WQP_Metadata" object.
# wqp queries require the statecode to be formatted as a 5-digit code, e.g. 'US:53' for Washington, so we need to use the get_wqp_statecode_from_state helper function to convert the state name to the correct format.
# I kept having trouble with the formatting, so I wrote a wee module to handle the conversion and ensure it's always correct.
sites_df_raw, metadata = wqp.what_sites(statecode=fips_code, siteType=site_type)
print(f"Found {len(sites_df_raw)} stream monitoring sites in state:{fips_code}, type:{site_type} for the period {start_date} to {end_date}.")

# 1. Save original raw DataFrame locally as CSV in raw landing zone first
raw_path = f"data/raw/{state_name.lower().replace(' ', '_')}_stream_sites.csv"
sites_df_raw.to_csv(raw_path, index=False)
print(f"Saved raw landing file to: {raw_path}")

Fetching stream monitoring sites in Washington...
Found 17936 stream monitoring sites in state:US:53, type:Stream for the period 01-01-2025 to 01-01-2025.
Saved raw landing file to: data/raw/washington_stream_sites.csv


# Raw -> Bronze

This is a "shallow" conversion. You take the raw CSV/JSON, add an ingestion_timestamp column, and save it as partitioned Parquet to get immediate read-performance gains.

In [19]:
from datetime import datetime, timezone
import pandas as pd

# Now we have the raw data in a DataFrame, lets add the ingestion_timestamp column, 
# which will be useful for tracking when the data was ingested and for debugging purposes
# if we need to trace back any issues to a specific ingestion batch. Then, we can write 
# it out as Parquet to the Bronze layer.
sites_df_bronze = sites_df_raw.copy()

# Add the ingestion timestamp column to the bronze DataFrame
sites_df_bronze["ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
sites_df_bronze["source_file"] = raw_path # add a column to track the source file for traceability
sites_df_bronze["source"] = "USGS WQP API" # add a column to track the source of the data
sites_df_bronze["query_metadata"] = str(metadata) # add a column to capture the query metadata for traceability and debugging

# Write out the validated records to the Bronze layer as Parquet
bronze_path = f"data/bronze/{state_name.lower().replace(' ', '_')}_stream_sites_validated.parquet"
pd.DataFrame(sites_df_bronze).to_parquet(bronze_path, index=False)
print(f"Saved {len(sites_df_bronze)} records to Bronze layer: {bronze_path}")

Saved 17936 records to Bronze layer: data/bronze/washington_stream_sites_validated.parquet


# Validate our choices in data file storage

There are a [number of benefits](https://www.databricks.com/blog/what-is-parquet) to using Parquet file format in our data pipeline, but for me the data compression we get is the most important.

In [7]:
# Compare the file size of the raw CSV and the Parquet in the Bronze layer to see the difference in size and to confirm that the Parquet file is smaller, which is one of the benefits of using Parquet for storage.

raw_size = os.path.getsize(raw_path)
bronze_size = os.path.getsize(bronze_path)
print(f"\nFile size comparison:")
print(f"Raw CSV file size: {raw_size / (1024 * 1024):.2f} MB")
print(f"Bronze Parquet file size: {bronze_size / (1024 * 1024):.2f} MB")
print(f"Ratio of Bronze Parquet size to Raw CSV size: {bronze_size / raw_size:.2%}")

assert bronze_size < raw_size, "Expected the Bronze Parquet file to be smaller than the raw CSV file, but it was not. Please check the files and the code for any issues."


File size comparison:
Raw CSV file size: 4.23 MB
Bronze Parquet file size: 0.82 MB
Ratio of Bronze Parquet size to Raw CSV size: 19.34%


# Which sorting column?

The effectiveness of parquet is dependent on the column chosen (see (Row Length encoding)[https://en.wikipedia.org/wiki/Run-length_encoding]). Lets do a little experiment: do the to_parquet() operation for all 37 sorted columns, and see what the best performance we can get is?

In [18]:
import pandas as pd
import os

def run_parquet_compression_experiment(df: pd.DataFrame, raw_csv_path: str, temp_dir: str = "data/tmp") -> pd.DataFrame:
    """
    Iterates through every column in a DataFrame, sorts the data by that column, 
    writes to Parquet, and measures the resulting compression metrics.
    """
    results = []
    
    # Create the temporary directory if it doesn't exist
    os.makedirs(temp_dir, exist_ok=True)
    
    # Get raw CSV size for the baseline ratio
    raw_size_bytes = os.path.getsize(raw_csv_path)
    
    # Create an unsorted baseline Parquet file to see how much SORTING specifically helps
    baseline_path = f"{temp_dir}/temp_baseline_unsorted.parquet"
    df.to_parquet(baseline_path, index=False)
    baseline_parquet_size = os.path.getsize(baseline_path)
    
    print(f"Starting Parquet compression experiment across {len(df.columns)} columns...")
    print(f"Raw CSV Size: {raw_size_bytes / (1024 * 1024):.2f} MB")
    print(f"Unsorted Parquet Size: {baseline_parquet_size / (1024 * 1024):.2f} MB\n")
    
    for col in df.columns:
        try:
            # 1. Sort the DataFrame by the current column
            sorted_df = df.sort_values(by=col)
            
            # 2. Write to a temporary Parquet file
            temp_path = f"{temp_dir}/temp_sorted_by_{col.replace('/', '_')}.parquet"
            sorted_df.to_parquet(temp_path, index=False)
            
            # 3. Measure file size
            file_size_bytes = os.path.getsize(temp_path)
            
            # 4. Calculate metrics
            csv_compression_ratio = file_size_bytes / raw_size_bytes
            
            # How much space did sorting save compared to just writing it unsorted?
            # Positive percentage means it shrank, negative means it bloated.
            sorting_savings_pct = (baseline_parquet_size - file_size_bytes) / baseline_parquet_size
            
            tmp_results =             {
                "Sort_Column": col,
                "File_Size_MB": file_size_bytes / (1024 * 1024),
                "CSV_Compression_Ratio": f"{csv_compression_ratio:.2%}",
                "Sorting_Savings_Pct": f"{sorting_savings_pct:.2%}",
                "Cardinality": df[col].nunique()
            }
            
            # 5. Record results
            results.append(tmp_results)
            
            # 6. Clean up the temp file immediately to save disk space
            os.remove(temp_path)
            
        except Exception as e:
            print(f"Skipping column '{col}' due to error: {e}")
            
    # Add the baseline entry explicitly so it shows up in the final table
    results.append({
        "Sort_Column": "BASELINE (Unsorted)",
        "File_Size_MB": baseline_parquet_size / (1024 * 1024),
        "CSV_Compression_Ratio": f"{(baseline_parquet_size / raw_size_bytes):.2%}",
        "Sorting_Savings_Pct": "0.00%",
        "Cardinality": "N/A"
    })
    
    # Clean up the baseline file
    os.remove(baseline_path)
    
    # Convert to DataFrame, sort by smallest file size, and return
    results_df = pd.DataFrame(results).sort_values(by="File_Size_MB", ascending=True).reset_index(drop=True)
    return results_df

# --- Run the Experiment ---
experiment_results = run_parquet_compression_experiment(sites_df_bronze, raw_path)

# round the MB file size to 2 decimal places for better readability in the final table
experiment_results["File_Size_MB"] = experiment_results["File_Size_MB"].round(2)

# Extract slices
top_3 = experiment_results.head(3)
baseline = experiment_results[experiment_results["Sort_Column"] == "BASELINE (Unsorted)"]
bottom_3 = experiment_results.tail(3)

# Concatenate the slices into a single DataFrame and remove any potential duplicates
summary_df = pd.concat([top_3, baseline, bottom_3]).drop_duplicates(subset=["Sort_Column"])

# Display the single, clean summary table
print("--- Compression Experiment Summary (Best, Baseline, Worst) ---")
display(summary_df)

# Save the full results to CSV
experiment_results.to_csv("data/raw/wpq_sites_compression_experiment_results.csv", index=False)

# Clean up the temp directory after the experiment
os.rmdir("data/tmp")

Starting Parquet compression experiment across 38 columns...
Raw CSV Size: 4.23 MB
Unsorted Parquet Size: 0.82 MB

--- Compression Experiment Summary (Best, Baseline, Worst) ---


,Sort_Column,File_Size_MB,CSV_Compression_Ratio,Sorting_Savings_Pct,Cardinality
0,OrganizationIdentifier,0.81,19.12%,1.11%,96
1,MonitoringLocationIdentifier,0.81,19.15%,0.99%,17936
2,HorizontalAccuracyMeasure/MeasureUnitCode,0.81,19.25%,0.46%,5
17,BASELINE (Unsorted),0.82,19.34%,0.00%,N/A
36,MonitoringLocationName,0.91,21.61%,-11.75%,15247
37,LongitudeMeasure,0.93,21.93%,-13.38%,14891
38,LatitudeMeasure,0.93,22.01%,-13.81%,15561


# Compression results

All things considered, this is a really tight range, with the best at 19.12%, worst at 22.01%.m

## Thoughts

* Baseline sorting was already pretty good. Its easy to make sorting savings WORSE!
* Parquet is doing dictionary encoding for the most common strings. This is likely where most of our savings is coming from. e.g. "NWIS" and "STORET" becoming 0, 1
* Due to sparsity of the dataset (many nan from well, lake columns), theres lots of empty space to compress. 

## Takeaway

Don't try to be a min/maxing power gamer. The data is just fine to write unsorted at this stage. Come back to it when you need to hot rod the pipeline, which will be never most likely.

The most important number is that 80% data storage savings.